Source: https://github.com/Project-MONAI/tutorials/blob/main/2d_segmentation/torch/unet_training_dict.py

In [ ]:
import logging
import os
import sys
from glob import glob
import time

# os.environ['KMP_DUPLICATE_LIB_OK'] = 'True'

import torch
from PIL import Image
from torch.utils.tensorboard import SummaryWriter

import monai
from monai.data import create_test_image_2d, list_data_collate, decollate_batch, DataLoader
from monai.inferers import sliding_window_inference
from monai.metrics import DiceMetric
from monai.transforms import (
    Activations,
    EnsureChannelFirstd,
    AsDiscrete,
    Compose,
    LoadImaged,
    RandCropByPosNegLabeld,
    RandRotate90d,
    ScaleIntensityd,
)
from monai.visualize import plot_2d_or_3d_image

c:\Users\pola3\miniconda3\envs\reg\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
data_dir = "data"
# monai.config.print_config()
logging.basicConfig(stream=sys.stdout, level=logging.INFO)

# create 40 random image, mask pairs in the target directory
print(f"generating synthetic data to '{data_dir}' (this may take a while)")
os.makedirs(data_dir, exist_ok=True)
for i in range(40):
    im, seg = create_test_image_2d(128, 128, num_seg_classes=1)
    Image.fromarray((im * 255).astype("uint8")).save(os.path.join(data_dir, f"img{i:d}.png"))
    Image.fromarray((seg * 255).astype("uint8")).save(os.path.join(data_dir, f"seg{i:d}.png"))

images = sorted(glob(os.path.join(data_dir, "img*.png")))
segs = sorted(glob(os.path.join(data_dir, "seg*.png")))
train_files = [{"img": img, "seg": seg} for img, seg in zip(images[:20], segs[:20])]
val_files = [{"img": img, "seg": seg} for img, seg in zip(images[-20:], segs[-20:])]

# define transforms for image and segmentation
train_transforms = Compose(
    [
        LoadImaged(keys=["img", "seg"]),
        EnsureChannelFirstd(keys=["img", "seg"]),
        ScaleIntensityd(keys=["img", "seg"]),
        RandCropByPosNegLabeld(
            keys=["img", "seg"], label_key="seg", spatial_size=[96, 96], pos=1, neg=1, num_samples=4
        ),
        RandRotate90d(keys=["img", "seg"], prob=0.5, spatial_axes=[0, 1]),
    ]
)
val_transforms = Compose(
    [
        LoadImaged(keys=["img", "seg"]),
        EnsureChannelFirstd(keys=["img", "seg"]),
        ScaleIntensityd(keys=["img", "seg"]),
    ]
)

# define dataset, data loader
check_ds = monai.data.Dataset(data=train_files, transform=train_transforms)
# use batch_size=2 to load images and use RandCropByPosNegLabeld to generate 2 x 4 images for network training
check_loader = DataLoader(check_ds, batch_size=2, num_workers=4, collate_fn=list_data_collate)
check_data = monai.utils.misc.first(check_loader)
print(check_data["img"].shape, check_data["seg"].shape)

# create a training data loader
train_ds = monai.data.Dataset(data=train_files, transform=train_transforms)
# use batch_size=2 to load images and use RandCropByPosNegLabeld to generate 2 x 4 images for network training
train_loader = DataLoader(
train_ds,
batch_size=2,
shuffle=True,
num_workers=4,
collate_fn=list_data_collate,
pin_memory=torch.cuda.is_available(),
)

# create a validation data loader
val_ds = monai.data.Dataset(data=val_files, transform=val_transforms)
val_loader = DataLoader(val_ds, batch_size=1, num_workers=4, collate_fn=list_data_collate)
dice_metric = DiceMetric(include_background=True, reduction="mean", get_not_nans=False)
post_trans = Compose([Activations(sigmoid=True), AsDiscrete(threshold=0.5)])

generating synthetic data to 'data' (this may take a while)
torch.Size([8, 1, 96, 96]) torch.Size([8, 1, 96, 96])


In [ ]:
from monai.networks.blocks import PatchEmbed
import torch.nn as nn
from mspe import MSPEPatchEmbedSwin

# customPatchEmbeddingBlock = PatchEmbed(patch_size=2, in_chans=1,
#                                     embed_dim=24, norm_layer=nn.LayerNorm,
#                                     spatial_dims=2)

# MSPE-SWIN embedding block
customPatchEmbeddingBlock = MSPEPatchEmbedSwin(patch_size=2, in_chans=1,
                                        embed_dim=24, norm_layer=nn.LayerNorm,
                                        spatial_dims=2, K=2, 
                                        resolutions=[64, 96, 128], img_size=96)
                                        

print(customPatchEmbeddingBlock, '\n')

device = torch.device("cuda")
model = monai.networks.nets.SwinUNETR( 
    in_channels=1,
    out_channels=1,
    spatial_dims=2,
    feature_size=24,     
    use_v2=True
).to(device)

model.swinViT.patch_embed = customPatchEmbeddingBlock.to(device)

MSPEPatchEmbedSwin(
  patch_size=(2, 2), in_chans=1, embed_dim=24, img_size=96, K=2, N=48, kernel_sizes=[(2, 2), (4, 4)], resolutions=[64, 96, 128]
  (norm): LayerNorm((24,), eps=1e-05, elementwise_affine=True)
  (patch_kernels): ModuleList(
    (0): Conv2d(1, 24, kernel_size=(2, 2), stride=(2, 2))
    (1): Conv2d(1, 24, kernel_size=(4, 4), stride=(2, 2))
  )
) 



SwinUNETR(
  (swinViT): SwinTransformer(
    (patch_embed): MSPEPatchEmbedSwin(
      patch_size=(2, 2), in_chans=1, embed_dim=24, img_size=96, K=2, N=48, kernel_sizes=[(2, 2), (4, 4)], resolutions=[64, 96, 128]
      (norm): LayerNorm((24,), eps=1e-05, elementwise_affine=True)
      (patch_kernels): ModuleList(
        (0): Conv2d(1, 24, kernel_size=(2, 2), stride=(2, 2))
        (1): Conv2d(1, 24, kernel_size=(4, 4), stride=(2, 2))
      )
    )
    (pos_drop): Dropout(p=0.0, inplace=False)
    (layers1): ModuleList(
      (0): BasicLayer(
        (blocks): ModuleList(
          (0-1): 2 x SwinTransformerBlock(
            (norm1): LayerNorm((24,), eps=1e-05, elementwise_affine=True)
            (attn): WindowAttention(
              (qkv): Linear(in_features=24, out_features=72, bias=True)
              (attn_drop): Dropout(p=0.0, inplace=False)
              (proj): Linear(in_features=24, out_features=24, bias=True)
              (proj_drop): Dropout(p=0.0, inplace=False)
         

In [ ]:

from mspe import img_resize
import torch.nn.functional as F

# MSPE embedding layers
mspe_embed = model.swinViT.patch_embed 

loss_function = monai.losses.DiceLoss(sigmoid=True)
optimizer = torch.optim.Adam(model.parameters(), 1e-4)

val_interval = 2
best_metric = -1
best_metric_epoch = -1
epoch_loss_values = list()
metric_values = list()
writer = SummaryWriter()
epochs = 10
lambda_reg = 1.0  # regularization weight 

for epoch in range(epochs):
    start = time.time()
    print("-" * 10)
    print(f"epoch {epoch + 1}/{epochs}")
    model.train()
    epoch_loss = 0
    step = 0
    for batch_data in train_loader:
        step += 1
        inputs, labels = batch_data["img"].to(device), batch_data["seg"].to(device)
        optimizer.zero_grad()

        # MSPE multi-res training 
        # sample K resolutions
        hw_list = mspe_embed.sample_resolutions()
        total_loss = torch.tensor(0.0, device=device)

        for k, hw in enumerate(hw_list):
            # resize both image and mask to resolution hw
            img_k = img_resize(inputs, hw)
            # nearest interpolation for masks to keep them binary
            seg_k = F.interpolate(labels, size=(hw, hw), mode="nearest")

            # forward through the full model at this resolution
            # that auto selects the best kernel for resolution hw
            outputs_k = model(img_k)

            # resize prediction back to original label size for loss computation
            if outputs_k.shape[-2:] != labels.shape[-2:]:
                outputs_k = F.interpolate(outputs_k, size=labels.shape[-2:], mode="bilinear", align_corners=False)

            total_loss = total_loss + loss_function(outputs_k, labels)

        # Original resolution forward pass + lambda regularization
        outputs_orig = model(inputs)
        total_loss = total_loss + lambda_reg* loss_function(outputs_orig, labels)

        total_loss.backward()
        optimizer.step()

        loss_val = total_loss.item()
        epoch_loss += loss_val
        epoch_len = len(train_ds) // train_loader.batch_size
        print(f"{step}/{epoch_len}, train_loss: {loss_val:.4f}")
        writer.add_scalar("train_loss", loss_val, epoch_len * epoch + step)

    epoch_loss /= step
    epoch_loss_values.append(epoch_loss)
    print(f"epoch {epoch + 1} average loss: {epoch_loss:.4f}")

    if (epoch + 1) % val_interval == 0:
        model.eval()
        with torch.no_grad():
            val_images = None
            val_labels = None
            val_outputs = None
            for val_data in val_loader:
                val_images, val_labels = val_data["img"].to(device), val_data["seg"].to(device)
                roi_size = (96, 96)
                sw_batch_size = 4
                val_outputs = sliding_window_inference(val_images, roi_size, sw_batch_size, model)
                val_outputs = [post_trans(i) for i in decollate_batch(val_outputs)]
                dice_metric(y_pred=val_outputs, y=val_labels)
            metric = dice_metric.aggregate().item()
            dice_metric.reset()
            metric_values.append(metric)
            if metric > best_metric:
                best_metric = metric
                best_metric_epoch = epoch + 1
                torch.save(model.state_dict(), "best_metric_model_segmentation2d_dict.pth")
                print("saved new best metric model")
            print(
                "current epoch: {} current mean dice: {:.4f} best mean dice: {:.4f} at epoch {}".format(
                    epoch + 1, metric, best_metric, best_metric_epoch
                )
            )
            writer.add_scalar("val_mean_dice", metric, epoch + 1)
            plot_2d_or_3d_image(val_images, epoch + 1, writer, index=0, tag="image")
            plot_2d_or_3d_image(val_labels, epoch + 1, writer, index=0, tag="label")
            plot_2d_or_3d_image(val_outputs, epoch + 1, writer, index=0, tag="output")

    end = time.time()
    print(f"epoch {epoch + 1} duration: {end - start:.2f} seconds")
print(f"train completed, best_metric: {best_metric:.4f} at epoch: {best_metric_epoch}")
writer.close()


----------
epoch 1/10
1/10, train_loss: 1.2738
2/10, train_loss: 1.2907
3/10, train_loss: 1.1935
4/10, train_loss: 1.1560
5/10, train_loss: 1.1678
6/10, train_loss: 1.1334
7/10, train_loss: 1.0570
8/10, train_loss: 0.9754
9/10, train_loss: 0.9620
10/10, train_loss: 0.9202
epoch 1 average loss: 1.1130
epoch 1 duration: 23.74 seconds
----------
epoch 2/10
1/10, train_loss: 0.8900
2/10, train_loss: 0.8868
3/10, train_loss: 0.8349
4/10, train_loss: 0.8308
5/10, train_loss: 0.7732
6/10, train_loss: 0.8003
7/10, train_loss: 0.7508
8/10, train_loss: 0.7452
9/10, train_loss: 0.7031
10/10, train_loss: 0.7132
epoch 2 average loss: 0.7928


c:\Users\pola3\miniconda3\envs\reg\Lib\site-packages\monai\inferers\utils.py:226: UserWarning: Using a non-tuple sequence for multidimensional indexing is deprecated and will be changed in pytorch 2.9; use x[tuple(seq)] instead of x[seq]. In pytorch 2.9 this will be interpreted as tensor index, x[torch.tensor(seq)], which will result either in an error or a different result (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\pytorch\torch\csrc\autograd\python_variable_indexing.cpp:353.)
  win_data = torch.cat([inputs[win_slice] for win_slice in unravel_slice]).to(sw_device)
c:\Users\pola3\miniconda3\envs\reg\Lib\site-packages\monai\inferers\utils.py:370: UserWarning: Using a non-tuple sequence for multidimensional indexing is deprecated and will be changed in pytorch 2.9; use x[tuple(seq)] instead of x[seq]. In pytorch 2.9 this will be interpreted as tensor index, x[torch.tensor(seq)], which will result either in an error or a different result (Triggered internally at C:\a

saved new best metric model
current epoch: 2 current mean dice: 0.9746 best mean dice: 0.9746 at epoch 2
epoch 2 duration: 36.38 seconds
----------
epoch 3/10
1/10, train_loss: 0.6723
2/10, train_loss: 0.6573
3/10, train_loss: 0.6433
4/10, train_loss: 0.6312
5/10, train_loss: 0.6553
6/10, train_loss: 0.6142
7/10, train_loss: 0.6515
8/10, train_loss: 0.5789
9/10, train_loss: 0.5665
10/10, train_loss: 0.5729
epoch 3 average loss: 0.6243
epoch 3 duration: 19.34 seconds
----------
epoch 4/10
1/10, train_loss: 0.5527
2/10, train_loss: 0.5470
3/10, train_loss: 0.5286
4/10, train_loss: 0.5479
5/10, train_loss: 0.5452
6/10, train_loss: 0.5331
7/10, train_loss: 0.5064
8/10, train_loss: 0.5278
9/10, train_loss: 0.5146
10/10, train_loss: 0.5191
epoch 4 average loss: 0.5322
saved new best metric model
current epoch: 4 current mean dice: 0.9862 best mean dice: 0.9862 at epoch 4
epoch 4 duration: 36.59 seconds
----------
epoch 5/10
1/10, train_loss: 0.5036
2/10, train_loss: 0.5160
3/10, train_loss: 

In [ ]:
# # start a typical PyTorch training

# loss_function = monai.losses.DiceLoss(sigmoid=True)
# optimizer = torch.optim.Adam(model.parameters(), 1e-3)

# val_interval = 2
# best_metric = -1
# best_metric_epoch = -1
# epoch_loss_values = list()
# metric_values = list()
# writer = SummaryWriter()
# epochs = 5

# for epoch in range(epochs):
#     start = time.time()
#     print("-" * 10)
#     print(f"epoch {epoch + 1}/{10}")
#     model.train()
#     epoch_loss = 0
#     step = 0
#     for batch_data in train_loader:
#         step += 1
#         inputs, labels = batch_data["img"].to(device), batch_data["seg"].to(device)
#         optimizer.zero_grad()
#         outputs = model(inputs)
#         loss = loss_function(outputs, labels)
#         loss.backward()
#         optimizer.step()
#         epoch_loss += loss.item()
#         epoch_len = len(train_ds) // train_loader.batch_size
#         print(f"{step}/{epoch_len}, train_loss: {loss.item():.4f}")
#         writer.add_scalar("train_loss", loss.item(), epoch_len * epoch + step)
#     epoch_loss /= step
#     epoch_loss_values.append(epoch_loss)
#     print(f"epoch {epoch + 1} average loss: {epoch_loss:.4f}")

#     if (epoch + 1) % val_interval == 0:
#         model.eval()
#         with torch.no_grad():
#             val_images = None
#             val_labels = None
#             val_outputs = None
#             for val_data in val_loader:
#                 val_images, val_labels = val_data["img"].to(device), val_data["seg"].to(device)
#                 roi_size = (96, 96)
#                 sw_batch_size = 4
#                 val_outputs = sliding_window_inference(val_images, roi_size, sw_batch_size, model)
#                 val_outputs = [post_trans(i) for i in decollate_batch(val_outputs)]
#                 # compute metric for current iteration
#                 dice_metric(y_pred=val_outputs, y=val_labels)
#             # aggregate the final mean dice result
#             metric = dice_metric.aggregate().item()
#             # reset the status for next validation round
#             dice_metric.reset()
#             metric_values.append(metric)
#             if metric > best_metric:
#                 best_metric = metric
#                 best_metric_epoch = epoch + 1
#                 torch.save(model.state_dict(), "best_metric_model_segmentation2d_dict.pth")
#                 print("saved new best metric model")
#             print(
#                 "current epoch: {} current mean dice: {:.4f} best mean dice: {:.4f} at epoch {}".format(
#                     epoch + 1, metric, best_metric, best_metric_epoch
#                 )
#             )
#             writer.add_scalar("val_mean_dice", metric, epoch + 1)
#             # plot the last model output as GIF image in TensorBoard with the corresponding image and label
#             plot_2d_or_3d_image(val_images, epoch + 1, writer, index=0, tag="image")
#             plot_2d_or_3d_image(val_labels, epoch + 1, writer, index=0, tag="label")
#             plot_2d_or_3d_image(val_outputs, epoch + 1, writer, index=0, tag="output")

#     end = time.time()
#     print(f"epoch {epoch + 1} duration: {end - start:.2f} seconds") 
# print(f"train completed, best_metric: {best_metric:.4f} at epoch: {best_metric_epoch}")
# writer.close()